# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a simple, transparent baseline for the **Refresh / Content Opportunity** lane.

The baseline is intentionally hand-written rather than learned by a model. The goal is to create a reasonable review queue that a later ML model must beat.

**Decision moment:** end of **2026-03-15**

- Past feature window: **2026-03-01 to 2026-03-15**
- Future outcome window: **2026-03-16 to 2026-03-31**
- Minimum usable GSC days in each half: **8**
- Later decline proxy: future impressions/day ≤ **80%** of past impressions/day

The later decline proxy is used only to **audit and evaluate** the baseline. It is never used to calculate the baseline score.

## Preliminaries

In [1]:
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

api = HfApi()
me = api.whoami(token=hf_token)
print("Authenticated as:", me["name"])

Authenticated as: ExoCeph


In [2]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token,
)

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token,
)

con = duckdb.connect()

FEATURE_START = "2026-03-01"
DECISION_DATE = "2026-03-15"
LABEL_START = "2026-03-16"
LABEL_END = "2026-03-31"
MIN_GSC_DAYS = 8
DECLINE_RATIO = 0.80

print("Warehouse ready.")
print("Feature window:", FEATURE_START, "to", DECISION_DATE)
print("Outcome window:", LABEL_START, "to", LABEL_END)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Warehouse ready.
Feature window: 2026-03-01 to 2026-03-15
Outcome window: 2026-03-16 to 2026-03-31


### Build the honest page-level table

One row represents one eligible client-content item. The baseline may use only information available by March 15. `is_declining` is included only so the signals and final ranking can be evaluated against the later outcome.

In [3]:
baseline_df = con.sql(
    f"""
    WITH past AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)::DOUBLE / COUNT(*) AS past_impressions_per_day,
            SUM(gsc_clicks)::DOUBLE / COUNT(*) AS past_clicks_per_day,
            SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS past_avg_position,
            SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS past_ctr,
            COUNT(*) AS past_available_days
        FROM read_parquet('{march_path}')
        WHERE
            report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
            AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(*) >= {MIN_GSC_DAYS} AND SUM(gsc_impressions) > 0
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)::DOUBLE / COUNT(*) AS future_impressions_per_day,
            COUNT(*) AS future_available_days
        FROM read_parquet('{march_path}')
        WHERE
            report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
            AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(*) >= {MIN_GSC_DAYS}
    )
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.past_impressions_per_day,
        p.past_clicks_per_day,
        p.past_avg_position,
        p.past_ctr,
        DATE_DIFF('day', c.content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
        c.content_type,
        c.main_intent,
        CASE
            WHEN f.future_impressions_per_day <= {DECLINE_RATIO} * p.past_impressions_per_day
            THEN 1 ELSE 0
        END AS is_declining
    FROM past AS p
    INNER JOIN future AS f
        ON p.client_hash_id = f.client_hash_id
       AND p.content_hash_id = f.content_hash_id
    INNER JOIN read_parquet('{content_path}') AS c
        ON p.client_hash_id = c.client_hash_id
       AND p.content_hash_id = c.content_hash_id
    WHERE
        c.content_created_date IS NOT NULL
        AND c.content_created_date <= DATE '{DECISION_DATE}'
    """
).df()

print("Eligible pages:", len(baseline_df))
print("Observed decline rate:", round(baseline_df["is_declining"].mean(), 3))
display(baseline_df.head())

Eligible pages: 103710
Observed decline rate: 0.311


,client_hash_id,content_hash_id,past_impressions_per_day,past_clicks_per_day,past_avg_position,past_ctr,content_age_days,content_type,main_intent,is_declining
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,28.600000,0.133333,4.386946,0.004662,380,keyword article,commercial,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1.636364,0.000000,4.833333,0.000000,380,keyword article,transactional,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,5.933333,0.000000,3.471910,0.000000,380,keyword article,transactional,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,41.866667,0.066667,5.265924,0.001592,380,keyword article,commercial,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,85.333333,0.600000,4.144531,0.007031,380,keyword article,informational,0


# 1. My rule and its reason codes

Before writing the rule, I check **two signals**.

- **Signal 1 — CTR relative to search position.** This is linked to the CTR-fix logic. Raw CTR is hard to interpret without knowing where a page ranks.
- **Signal 2 — content age / staleness.** I test whether older content is actually more likely to show the later decline proxy.

I do not force either signal into the rule unless the observed bucket tables support it.

### Signal 1A — CTR changes with search position

In [4]:
position_audit = baseline_df.copy()
position_audit["position_bucket"] = pd.cut(
    position_audit["past_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    right=True,
)

position_table = (
    position_audit
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("past_ctr", "median"),
        mean_ctr=("past_ctr", "mean"),
        median_impressions_per_day=("past_impressions_per_day", "median"),
        observed_decline_rate=("is_declining", "mean"),
    )
    .reset_index()
)

position_table["median_ctr_pct"] = 100 * position_table["median_ctr"]
position_table["mean_ctr_pct"] = 100 * position_table["mean_ctr"]
position_table["observed_decline_rate_pct"] = 100 * position_table["observed_decline_rate"]

display(position_table[[
    "position_bucket", "n", "median_ctr_pct", "mean_ctr_pct",
    "median_impressions_per_day", "observed_decline_rate_pct"
]])

,position_bucket,n,median_ctr_pct,mean_ctr_pct,median_impressions_per_day,observed_decline_rate_pct
0,1-3,10968,0.184162,0.370593,50.266667,31.345733
1,4-10,47426,0.112160,0.341946,29.200000,32.404166
2,11-20,18920,0.000000,0.260514,14.464103,28.768499
3,21-50,20768,0.000000,0.157045,13.466667,32.516371
4,51+,5619,0.000000,0.059605,4.357143,21.694252


**Observed interpretation:** CTR falls as average search position gets worse, so one global CTR threshold would be misleading. Position by itself does not show a simple monotonic relationship with the later decline proxy.

Next I compare CTR against peers **within similar top-10 position groups**.

### Signal 1B — weak CTR relative to similar-ranking pages

In [5]:
ctr_audit = baseline_df[baseline_df["past_avg_position"] <= 10].copy()

ctr_audit["position_group"] = pd.cut(
    ctr_audit["past_avg_position"],
    bins=[0, 3, 10],
    labels=["1-3", "4-10"],
)

ctr_medians = (
    ctr_audit
    .groupby("position_group", observed=True)["past_ctr"]
    .median()
)

ctr_audit["peer_median_ctr"] = ctr_audit["position_group"].map(ctr_medians).astype(float)

ctr_audit["ctr_bucket"] = np.select(
    [
        ctr_audit["past_ctr"] == 0,
        ctr_audit["past_ctr"] < ctr_audit["peer_median_ctr"],
    ],
    ["ZERO_CTR", "BELOW_PEER_MEDIAN"],
    default="AT_OR_ABOVE_PEER_MEDIAN",
)

ctr_signal_table = (
    ctr_audit
    .groupby(["position_group", "ctr_bucket"], observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("past_ctr", "median"),
        median_impressions_per_day=("past_impressions_per_day", "median"),
        observed_decline_rate=("is_declining", "mean"),
    )
    .reset_index()
)

ctr_signal_table["median_ctr_pct"] = 100 * ctr_signal_table["median_ctr"]
ctr_signal_table["observed_decline_rate_pct"] = 100 * ctr_signal_table["observed_decline_rate"]

display(ctr_signal_table[[
    "position_group", "ctr_bucket", "n", "median_ctr_pct",
    "median_impressions_per_day", "observed_decline_rate_pct"
]])

,position_group,ctr_bucket,n,median_ctr_pct,median_impressions_per_day,observed_decline_rate_pct
0,1-3,AT_OR_ABOVE_PEER_MEDIAN,5486,0.495050,83.833333,24.480496
1,1-3,BELOW_PEER_MEDIAN,1611,0.119095,124.800000,38.981999
2,1-3,ZERO_CTR,3871,0.000000,12.600000,37.897184
3,4-10,AT_OR_ABOVE_PEER_MEDIAN,23713,0.427960,57.066667,27.786446
4,4-10,BELOW_PEER_MEDIAN,2881,0.076104,145.066667,44.463728
5,4-10,ZERO_CTR,20832,0.000000,9.266667,35.992704


**Signal 1 verdict — CONFIRMED.**

Among top-10 pages, below-peer CTR is associated with a substantially higher later observed decline rate than CTR at or above the peer median.

Observed in this slice:

- Positions 1–3: below-peer CTR ≈ **39.0%** decline vs at/above peer median ≈ **24.5%**
- Positions 4–10: below-peer CTR ≈ **44.5%** decline vs at/above peer median ≈ **27.8%**

Zero-CTR pages often have much lower visibility, so the baseline should not prioritize zero CTR alone. I use **good position + weak peer-relative CTR + meaningful visibility** as the core rule.

These are observed directional relationships, not causal evidence that changing CTR will prevent decline.

### Signal 2 — content age / staleness

In [6]:
age_audit = baseline_df.copy()
age_audit["age_bucket"] = pd.cut(
    age_audit["content_age_days"],
    bins=[-np.inf, 90, 180, 365, 730, np.inf],
    labels=["0-90", "91-180", "181-365", "366-730", "731+"],
)

age_signal_table = (
    age_audit
    .groupby("age_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_age_days=("content_age_days", "median"),
        median_impressions_per_day=("past_impressions_per_day", "median"),
        median_ctr=("past_ctr", "median"),
        observed_decline_rate=("is_declining", "mean"),
    )
    .reset_index()
)

age_signal_table["median_ctr_pct"] = 100 * age_signal_table["median_ctr"]
age_signal_table["observed_decline_rate_pct"] = 100 * age_signal_table["observed_decline_rate"]

display(age_signal_table[[
    "age_bucket", "n", "median_age_days", "median_impressions_per_day",
    "median_ctr_pct", "observed_decline_rate_pct"
]])

,age_bucket,n,median_age_days,median_impressions_per_day,median_ctr_pct,observed_decline_rate_pct
0,0-90,30570,47.0,23.933333,0.031671,34.357213
1,91-180,21227,144.0,19.800000,0.000000,37.928110
2,181-365,40319,244.0,21.933333,0.000000,26.540837
3,366-730,11594,396.0,14.076923,0.000000,25.582198


**Signal 2 verdict — MIXED.**

The observed later decline rate does **not** rise consistently with content age.

Observed in this slice:

- 0–90 days: ≈ **34.4%**
- 91–180 days: ≈ **37.9%**
- 181–365 days: ≈ **26.5%**
- 366–730 days: ≈ **25.6%**

Age alone is therefore not a reliable monotonic refresh signal in this slice, so I do **not** use it as a primary component of the baseline score.

### Final baseline rule

I prioritize pages that:

1. have an interpretable average search position between **1 and 10**,
2. have CTR below the median CTR of pages in the same position group,
3. and have meaningful search visibility.

The score combines:

**peer-relative CTR deficit × visibility percentile × position weight**

Reason codes:

- `CTR_GAP_HIGH_VISIBILITY` — CTR is weak relative to peers and visibility is high.
- `CTR_GAP` — CTR is weak relative to peers but visibility is lower.
- `POSITION_DATA_CHECK` — reported average position is below 1, so I do not trust the CTR-vs-position interpretation for scoring.
- `NO_CTR_OPPORTUNITY` — the baseline does not see a supported CTR-based opportunity.

Actions: `REVIEW_CTR` or `MONITOR`.

The baseline score uses only past-time information. The later decline label is not an input.

# 2. Build the ranked queue (writes the CSV)

In [7]:
queue = baseline_df.copy()

queue["valid_ctr_position"] = (
    (queue["past_avg_position"] >= 1)
    & (queue["past_avg_position"] <= 10)
)

queue["position_group"] = pd.cut(
    queue["past_avg_position"],
    bins=[1, 3, 10],
    labels=["1-3", "4-10"],
    include_lowest=True,
)

peer_ctr = (
    queue[queue["valid_ctr_position"]]
    .groupby("position_group", observed=True)["past_ctr"]
    .median()
)

print("Peer CTR medians:")
print(peer_ctr)

Peer CTR medians:
position_group
1-3     0.002093
4-10    0.001122
Name: past_ctr, dtype: float64


In [8]:
queue["peer_median_ctr"] = queue["position_group"].map(peer_ctr).astype(float)

queue["ctr_deficit_ratio"] = (
    (queue["peer_median_ctr"] - queue["past_ctr"])
    / queue["peer_median_ctr"]
)
queue["ctr_deficit_ratio"] = (
    queue["ctr_deficit_ratio"]
    .clip(lower=0, upper=1)
    .fillna(0)
)

queue["visibility_percentile"] = (
    queue["past_impressions_per_day"].rank(pct=True)
)

queue["position_weight"] = np.select(
    [
        queue["valid_ctr_position"] & (queue["past_avg_position"] <= 3),
        queue["valid_ctr_position"] & (queue["past_avg_position"] <= 10),
    ],
    [1.0, 0.85],
    default=0.0,
)

queue["baseline_action_score"] = (
    100
    * queue["ctr_deficit_ratio"]
    * queue["visibility_percentile"]
    * queue["position_weight"]
)

In [9]:
queue["reason_code"] = np.select(
    [
        queue["past_avg_position"] < 1,
        (queue["baseline_action_score"] > 0)
        & (queue["visibility_percentile"] >= 0.75),
        queue["baseline_action_score"] > 0,
    ],
    [
        "POSITION_DATA_CHECK",
        "CTR_GAP_HIGH_VISIBILITY",
        "CTR_GAP",
    ],
    default="NO_CTR_OPPORTUNITY",
)

queue["action"] = np.where(
    queue["baseline_action_score"] > 0,
    "REVIEW_CTR",
    "MONITOR",
)

queue = queue.sort_values(
    "baseline_action_score",
    ascending=False,
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "action",
    "past_avg_position",
    "past_ctr",
    "past_impressions_per_day",
]

queue[output_cols].to_csv(OUTPUT_PATH, index=False)

print("Queue rows:", len(queue))
print("Position-data-check rows:", (queue["reason_code"] == "POSITION_DATA_CHECK").sum())
print("Saved:", OUTPUT_PATH)
display(queue[output_cols].head(20))

Queue rows: 103710
Position-data-check rows: 947
Saved: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action,past_avg_position,past_ctr,past_impressions_per_day
0,1,client_1a730cb2640a1abf,content_d61fc394d10cba41,99.517886,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,2.992886,0.000000,1325.285714
1,2,client_fef1a8f436438636,content_66bf45eb0c5bb550,98.550767,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,1.793227,0.000000,763.800000
2,3,client_20259bd6705d81d4,content_416fca7aa8ace93b,97.892199,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,1.538512,0.000000,612.800000
3,4,client_73cda7b4e4f265ea,content_41d18608d90de375,97.619323,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,1.344031,0.000000,571.266667
4,5,client_62f4a7e64f5e0096,content_03f33581fac04b10,97.487224,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,2.022749,0.000000,553.866667
5,6,client_e547b89c05043229,content_713b157e9c77690a,96.855655,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,2.724264,0.000000,483.461538
6,7,client_62f4a7e64f5e0096,content_cb936ac68efb3702,96.392344,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,2.984406,0.000000,440.333333
7,8,client_62f4a7e64f5e0096,content_eddf71c8b99e10ca,96.187446,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,1.478820,0.000000,424.933333
8,9,client_62f4a7e64f5e0096,content_8a3fdc6f7006ad25,96.107415,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,2.038168,0.000000,419.200000
9,10,client_73cda7b4e4f265ea,content_0e034bf052f81ee8,95.773310,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR,2.479630,0.000000,396.000000


# 3. Top-20 review

For each of the top 20, I record the action, reason code, why it ranked highly, a confidence note, and what could make the recommendation wrong.

The queue is decision support, not an automatic fix system.

In [10]:
review_top20 = queue.head(20).copy()

review_top20["why_ranked"] = review_top20.apply(
    lambda r:
        f"Ranks around position {r['past_avg_position']:.2f}, "
        f"gets {r['past_impressions_per_day']:.1f} impressions/day, "
        f"but CTR is only {100*r['past_ctr']:.3f}%.",
    axis=1,
)

review_top20["confidence_note"] = np.where(
    review_top20["past_impressions_per_day"] >= 250,
    "Higher confidence: strong visibility makes the CTR gap meaningful.",
    "Moderate confidence: CTR gap exists, but visibility is lower.",
)

review_top20["what_could_make_it_wrong"] = (
    "Low CTR could reflect query mix, SERP features, reporting behavior, "
    "or other factors not captured by this baseline."
)

review_cols = [
    "rank",
    "content_hash_id",
    "baseline_action_score",
    "action",
    "reason_code",
    "past_avg_position",
    "past_ctr",
    "past_impressions_per_day",
    "why_ranked",
    "confidence_note",
    "what_could_make_it_wrong",
]

display(review_top20[review_cols])

,rank,content_hash_id,baseline_action_score,action,reason_code,past_avg_position,past_ctr,past_impressions_per_day,why_ranked,confidence_note,what_could_make_it_wrong
0,1,content_d61fc394d10cba41,99.517886,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,2.992886,0.000000,1325.285714,"Ranks around position 2.99, gets 1325.3 impres...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
1,2,content_66bf45eb0c5bb550,98.550767,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,1.793227,0.000000,763.800000,"Ranks around position 1.79, gets 763.8 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
2,3,content_416fca7aa8ace93b,97.892199,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,1.538512,0.000000,612.800000,"Ranks around position 1.54, gets 612.8 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
3,4,content_41d18608d90de375,97.619323,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,1.344031,0.000000,571.266667,"Ranks around position 1.34, gets 571.3 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
4,5,content_03f33581fac04b10,97.487224,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,2.022749,0.000000,553.866667,"Ranks around position 2.02, gets 553.9 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
5,6,content_713b157e9c77690a,96.855655,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,2.724264,0.000000,483.461538,"Ranks around position 2.72, gets 483.5 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
6,7,content_cb936ac68efb3702,96.392344,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,2.984406,0.000000,440.333333,"Ranks around position 2.98, gets 440.3 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
7,8,content_eddf71c8b99e10ca,96.187446,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,1.478820,0.000000,424.933333,"Ranks around position 1.48, gets 424.9 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
8,9,content_8a3fdc6f7006ad25,96.107415,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,2.038168,0.000000,419.200000,"Ranks around position 2.04, gets 419.2 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."
9,10,content_0e034bf052f81ee8,95.773310,REVIEW_CTR,CTR_GAP_HIGH_VISIBILITY,2.479630,0.000000,396.000000,"Ranks around position 2.48, gets 396.0 impress...",Higher confidence: strong visibility makes the...,"Low CTR could reflect query mix, SERP features..."


**Top-20 review summary:** After excluding anomalous position values below 1 from CTR scoring, the highest-ranked recommendations are pages with strong reported search positions, meaningful visibility, and zero or near-zero CTR.

These are plausible `REVIEW_CTR` candidates, but low CTR can also reflect query mix, SERP features, reporting behavior, or other unobserved factors. The rule therefore ranks pages for human review rather than applying an automatic content change.

# 4. Weak picks + leakage check

I now attack the baseline in two ways:

1. inspect high-scoring recommendations that **did not** show the later decline proxy,
2. verify that no future-window, target-derived, or product-rule fields entered the score.

I also investigate the unusual position values below 1 that appeared in the first version of the queue.

In [11]:
bad_picks = (
    queue[
        (queue["action"] == "REVIEW_CTR")
        & (queue["is_declining"] == 0)
    ]
    .sort_values("baseline_action_score", ascending=False)
    .head(10)
)

display(bad_picks[[
    "rank",
    "content_hash_id",
    "baseline_action_score",
    "past_avg_position",
    "past_ctr",
    "past_impressions_per_day",
    "is_declining",
]])

,rank,content_hash_id,baseline_action_score,past_avg_position,past_ctr,past_impressions_per_day,is_declining
0,1,content_d61fc394d10cba41,99.517886,2.992886,0.0,1325.285714,0
1,2,content_66bf45eb0c5bb550,98.550767,1.793227,0.0,763.800000,0
5,6,content_713b157e9c77690a,96.855655,2.724264,0.0,483.461538,0
6,7,content_cb936ac68efb3702,96.392344,2.984406,0.0,440.333333,0
11,12,content_df5abe1755bf02f9,95.441616,2.894960,0.0,377.000000,0
13,14,content_f5a7a2559d483a54,93.638029,2.496193,0.0,293.000000,0
18,19,content_b154f6c2652cfeb9,92.492045,2.838794,0.0,256.400000,0
19,20,content_3c7933bff4e48d0d,92.488188,1.678370,0.0,256.333333,0
20,21,content_b22ec1b972c8ecaf,92.172886,2.615426,0.0,247.200000,0
21,22,content_2c53ca9a79da7dfe,91.479124,2.324732,0.0,229.933333,0


These are useful weak picks: the rule saw a plausible CTR opportunity, but the page did not show the later visibility-decline proxy.

That does not automatically mean the CTR recommendation is false. Weak CTR and future visibility decline are related but not identical outcomes.

### Evaluate the hand-written baseline

In [12]:
for k in [10, 20, 50, 100]:
    top_k = queue.head(k)
    precision = top_k["is_declining"].mean()
    print(
        f"Precision@{k}: "
        f"{precision:.3f} "
        f"({int(top_k['is_declining'].sum())}/{k})"
    )

print("\nOverall decline rate:", round(queue["is_declining"].mean(), 3))

Precision@10: 0.600 (6/10)
Precision@20: 0.600 (12/20)
Precision@50: 0.580 (29/50)
Precision@100: 0.540 (54/100)

Overall decline rate: 0.311


**Observed baseline performance in this run:**

- Precision@10 = **0.600** (6/10)
- Precision@20 = **0.600** (12/20)
- Precision@50 = **0.580** (29/50)
- Precision@100 = **0.540** (54/100)
- Overall observed decline rate = **0.311**

The rule therefore concentrates more later-declining pages near the top of the queue than the overall population rate. This is a useful benchmark for the later ML model, not proof of causation.

### Position-data quality check

In [13]:
print(baseline_df["past_avg_position"].describe())

position_lt_1_count = int((baseline_df["past_avg_position"] < 1).sum())
print("\nPages with avg position < 1:", position_lt_1_count)

display(
    baseline_df.loc[
        baseline_df["past_avg_position"] < 1,
        ["past_avg_position", "past_impressions_per_day", "past_ctr"],
    ].head(10)
)

count    103710.000000
mean         15.355319
std          16.197242
min           0.000000
25%           4.787234
50%           8.504637
75%          20.382856
max         119.429688
Name: past_avg_position, dtype: float64

Pages with avg position < 1: 947


,past_avg_position,past_impressions_per_day,past_ctr
47,0.000000,4.846154,0.000000
50,0.248663,24.933333,0.000000
137,0.961643,342.400000,0.003115
250,0.152866,15.700000,0.000000
348,0.722222,8.400000,0.007937
688,0.000000,2.666667,0.000000
1308,0.355584,55.642857,0.000000
1466,0.841584,7.769231,0.000000
2513,0.107143,2.545455,0.000000
2526,0.476190,8.400000,0.000000


In this slice, **947 rows** have reported weighted average position below 1. Because this is unusual for the normal search-position interpretation used by the CTR rule, I do not treat these values as exceptionally strong rankings.

Instead, they receive `POSITION_DATA_CHECK` and a score of zero.

### Inspect one suspicious raw record

In [14]:
suspicious_candidates = (
    baseline_df[baseline_df["past_avg_position"] < 1]
    .sort_values("past_impressions_per_day", ascending=False)
)

if len(suspicious_candidates) > 0:
    example = suspicious_candidates.iloc[0]
    example_client = example["client_hash_id"]
    example_content = example["content_hash_id"]

    position_debug = con.sql(
        f"""
        SELECT
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_sum_position,
            gsc_avg_position,
            gsc_data_available
        FROM read_parquet('{march_path}')
        WHERE
            client_hash_id = '{example_client}'
            AND content_hash_id = '{example_content}'
            AND report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ORDER BY report_date
        """
    ).df()

    display(position_debug)

    weighted_position = (
        position_debug["gsc_sum_position"].sum()
        / position_debug["gsc_impressions"].sum()
    )
    mean_daily_position = position_debug["gsc_avg_position"].mean()

    print("Weighted position:", weighted_position)
    print("Mean daily gsc_avg_position:", mean_daily_position)

    display(position_debug[[
        "gsc_impressions",
        "gsc_sum_position",
        "gsc_avg_position",
    ]].describe())
else:
    print("No position < 1 rows found.")

,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,gsc_data_available
0,2026-03-01,28947,0,65,0.002245,True
1,2026-03-02,28973,0,9,0.000311,True
2,2026-03-03,24233,0,7706,0.317996,True
3,2026-03-04,1580,0,629,0.398101,True
4,2026-03-05,2,0,10,5.000000,True
5,2026-03-06,5,0,108,21.600000,True
6,2026-03-07,1,0,6,6.000000,True
7,2026-03-08,5,0,29,5.800000,True
8,2026-03-09,4,0,58,14.500000,True
9,2026-03-10,6,0,35,5.833333,True


Weighted position: 0.10527383851406198
Mean daily gsc_avg_position: 8.607910233598517


,gsc_impressions,gsc_sum_position,gsc_avg_position
count,15.000000,15.000000,15.000000
mean,5584.800000,587.933333,8.607910
std,11336.567036,1975.299705,12.407646
min,1.000000,6.000000,0.000311
25%,2.500000,10.000000,1.532384
50%,5.000000,32.000000,5.000000
75%,793.000000,80.500000,8.333333
max,28973.000000,7706.000000,48.000000


The raw daily inspection shows that the below-1 values originate in the warehouse itself rather than from my weighted-position formula. I therefore leave the source values unchanged but remove them from CTR scoring.

This is a data-quality / metric-definition safeguard, not a claim that the warehouse values are necessarily wrong.

### Leakage check

In [15]:
BASELINE_INPUTS = [
    "past_avg_position",
    "past_ctr",
    "past_impressions_per_day",
]

blocked_terms = [
    "future",
    "declin",
    "label",
    "target",
    "quick_win",
    "health_score",
    "refresh_candidate",
]

blocked_hits = [
    feature
    for feature in BASELINE_INPUTS
    if any(term in feature.lower() for term in blocked_terms)
]

print("Baseline score inputs:", BASELINE_INPUTS)
print("Blocked/leaky inputs found:", blocked_hits)

assert len(blocked_hits) == 0
print("Baseline leakage check passed.")

Baseline score inputs: ['past_avg_position', 'past_ctr', 'past_impressions_per_day']
Blocked/leaky inputs found: []
Baseline leakage check passed.


In [16]:
print(
    "Position-data-check rows:",
    (queue["reason_code"] == "POSITION_DATA_CHECK").sum()
)

assert not (
    queue.loc[
        queue["past_avg_position"] < 1,
        "baseline_action_score"
    ] > 0
).any()

print("Rows with position < 1 receive no CTR score.")

Position-data-check rows: 947
Rows with position < 1 receive no CTR score.


### Final weak-pick / leakage conclusion

The baseline is transparent but imperfect. Some high-scoring CTR-review candidates do not later decline because poor CTR and later visibility decline are not the same outcome.

I also found a position-metric edge case: 947 rows have reported average position below 1. The raw records show that these values come from the warehouse rather than my aggregation formula. I therefore keep them unchanged, exclude them from CTR scoring, and label them `POSITION_DATA_CHECK`.

The final score uses only:

- `past_avg_position`
- `past_ctr`
- `past_impressions_per_day`

No future-window, target-derived, or existing product-rule fields are used in the baseline score.

# Self-check

In [17]:
assert len(queue) == len(baseline_df)
assert OUTPUT_PATH.exists()

assert queue["baseline_action_score"].notna().all()
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()

assert not (
    queue.loc[
        queue["past_avg_position"] < 1,
        "baseline_action_score"
    ] > 0
).any()

assert len(blocked_hits) == 0

print("Self-check passed.")
print("Notebook population:", len(queue))
print("Output CSV:", OUTPUT_PATH)
print("Top-20 rows available:", len(review_top20))

Self-check passed.
Notebook population: 103710
Output CSV: work/outputs/baseline_action_score.csv
Top-20 rows available: 20


### Submission notes

- Every section above includes both reasoning and code.
- The notebook is designed to run top-to-bottom.
- Only pseudonymous IDs are shown; no client names, URLs, or private queries are included.
- Claims use careful language such as **observed**, **measured**, **directional**, and **decision-support**.
- The notebook writes `work/outputs/baseline_action_score.csv`.
- Commit this notebook under `work/notebooks/w04_baseline_score.ipynb`.